# Lecture 7 - matplotlib and xarray

**Lecture Topic**: Plotting with matplotlib and pandas, then labelled multi-dimensional data with xarray

## Announcements

- Project proposals (Milestone 1) are due next Friday, Oct 16. Please let me know by Wednesday if you're still looking for a group.
- Monday's and Wednesday's lectures are on Zoom, and there is no class next Friday.
- ClassBuzz poll for today: `RV7D`

## Recap

In the last lecture, we covered the following topics:
 - The course project, and Milestone 1
 - Summary statistics and group-by
 - Cleaning data: renaming, dates, new columns and sorting
 - Merging (and checking the number of rows afterwards)
 - Silent failure modes: alignment, types and chained assignment

Today, we will cover the following topics:
 - Why we visualize data
 - matplotlib: figures, axes and subplots
 - Plotting with pandas
 - xarray: labelled multi-dimensional data
 - Indexing, plotting and computing with xarray

## Why is data visualization necessary?

So far, we have focused a lot on summarizing the data using statistics. However, summary statistics are not enough to understand the distribution - there are many possible distributions for a given set of summary statistics. Data visualization is necessary to truly understand the distribution:

<div style="text-align: center; margin-top: -10px;">
<img width="50%" src="https://raw.githubusercontent.com/stefmolin/data-morph/main/docs/_static/panda-to-star-eased.gif" alt="Data Morph: panda to star" style="min-width: 300px; margin-bottom: -10px;"/>
<div style="margin: auto 26%;"><small><em>A set of points forming a panda can also form a star without any significant changes to the summary statistics displayed above. (source: <a href="https://github.com/stefmolin/data-morph">Data Morph</a>)</em></small></div>
</div>

This matters even more when an agent is doing the analysis. A plot is one of the quickest ways to check that the numbers it gives you are sensible.

## An introduction to matplotlib

Matplotlib is a Python package used widely throughout the scientific Python community to produce high quality 2D publication graphics. pandas and xarray both use it under the hood.

It is a general convention to import its **`pyplot`** module as **`plt`**:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## The matplotlib figure

At the heart of **every** matplotlib plot is the "Figure" object. The "Figure" object is the top level concept that can be drawn to one of the many output formats, or simply just to screen.

By far the most useful thing on a figure is the "Axes". The Axes represents the "data space" of a typical plot, with an x and y axis, tick labels and tick marks.

The easiest way to make both is with `plt.subplots()`:

In [ ]:
x = np.linspace(0, 2 * np.pi, 100)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, np.sin(x), label='sin(x)')
ax.plot(x, np.cos(x), label='cos(x)', linestyle='--')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Some trigonometry')
ax.legend()

### Multiple axes on the same figure (aka subplots)

We can ask `plt.subplots()` for a grid of axes, and get them back as an array:

In [ ]:
taxis = pd.read_csv('../data/2019_Yellow_Taxi_Trip_Data.csv')

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(taxis.fare_amount, bins=50)
axes[0].set_xlabel('Fare ($)')
axes[1].scatter(taxis.trip_distance, taxis.fare_amount, s=2, alpha=0.3)
axes[1].set_xlabel('Distance (miles)')
axes[1].set_ylabel('Fare ($)')
fig.tight_layout()  # handle overlaps

Question: what's going on with the horizontal line of points on the right?

Those are fixed-price trips: JFK airport to Manhattan was a flat $52 in 2019. Again, something you'd never see in `describe()`.

### Quick quiz

An agent wrote this to label the two histograms. Which panel ends up with the title?

 - A. The left one
 - B. The right one
 - C. Both
 - D. Neither, it goes above the whole figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(taxis.fare_amount, bins=50)
axes[1].hist(taxis.tip_amount, bins=50)
plt.title('Fares')
plt.xlabel('$')

`plt.title()` and friends act on the *current* axes, which is whichever one was used last. So the fare histogram has no label, and the tips are titled "Fares".

In general, try to stick to either the `ax.set_...` methods or the `plt.` functions rather than mixing the two, and be especially careful with `plt.` functions when you have subplots.

### Further plot types

Matplotlib comes with a huge variety of different plot types. The [gallery](https://matplotlib.org/stable/gallery/index.html) is the best place to find them (and agents are very good at writing this kind of code).

In [ ]:
lon = np.linspace(-180, 180, 60)
lat = np.linspace(-90, 90, 30)
lon2d, lat2d = np.meshgrid(lon, lat)
data = np.cos(3 * np.deg2rad(lon2d)) + np.sin(2 * np.deg2rad(lat2d))

fig, axes = plt.subplots(1, 3, figsize=(10, 2.5))
axes[0].contourf(lon, lat, data)
axes[1].bar(lon, data.sum(axis=0), width=np.diff(lon)[0])
mesh = axes[2].pcolormesh(lon, lat, data)
fig.colorbar(mesh, ax=axes[2], label='Some function [units]')
fig.tight_layout()

### Saving your plots

You can save a figure using **`fig.savefig`**. The format of the file is inferred from the file extension:

```python
fig.savefig('my_plot.png', dpi=150)
fig.savefig('my_plot.pdf')
```

## Plotting with pandas

We can create a variety of visualizations directly from a DataFrame or Series using the `plot()` method, which uses matplotlib under the hood:

In [ ]:
taxis.groupby('payment_type').fare_amount.mean().plot(kind='bar', ylabel='Mean fare ($)', rot=0)

The `plot()` method returns a matplotlib `Axes`, so it can be modified further (e.g., to add reference lines, annotations, labels, etc.), or you can pass in an `ax=` you made yourself:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
taxis.fare_amount.plot(kind='hist', bins=50, ax=ax)
ax.axvline(taxis.fare_amount.median(), color='k', linestyle='--', label='median')
ax.legend()

*Tip: [Seaborn](https://seaborn.pydata.org/) builds on matplotlib and pandas, and is very handy for statistical plots of long-format data.*

## Introduction to xarray

<img src="https://docs.xarray.dev/en/stable/_static/Xarray_Logo_RGB_Final.svg" align="center" width="30%">

pandas is great for tabular data, but a lot of data has more dimensions than that: climate model output, satellite images, medical scans, experiments repeated over many parameters...

xarray brings the labels from pandas to multi-dimensional NumPy arrays.

## Data structures

Xarray has two main data structures:

1. `DataArray`s, which wrap an underlying array (usually NumPy) along with its metadata: dimension names, coordinates and attributes
1. `Dataset`s, which are dictionary-like containers of DataArrays that share coordinates

<img src="https://docs.xarray.dev/en/stable/_images/dataset-diagram.png" align="center" width="50%">

Let's open some (projected) global surface temperature data:

In [ ]:
import xarray as xr

ds = xr.open_dataset('../data/global_temparature.nc ')
ds = ds.drop_vars('quantile')  # a leftover coordinate we don't need
ds

Xarray objects have convenient HTML representations to give an overview of what we're working with. This is a netCDF file, a very common format for gridded data.

## What's in a DataArray?

*data + (a lot of) metadata*. We can pull a DataArray out of a Dataset with dictionary syntax, `ds['tas']`, or "dot" notation:

In [ ]:
ds.tas.dims

In [ ]:
ds.tas.coords

It is useful to think of the values in these coordinate variables as axis "labels" such as "tick labels" in a figure. These are the locations on a grid at which you have data.

### Attributes

`.attrs` is a dictionary for anything else you'd like to record, like the units or where the data came from:

In [ ]:
ds.tas.attrs

This one is empty! I happen to know that this is the change in temperature (in °C) since pre-industrial times, but nothing in the file tells you that. If you hand this to an agent, it will have to guess.

Let's fix that:

In [ ]:
ds.tas.attrs['units'] = 'degC'
ds.tas.attrs['long_name'] = 'Surface temperature change since pre-industrial'

### Underlying data

`.data` contains the NumPy array storing the actual values:

In [ ]:
type(ds.tas.data), ds.tas.data.shape

## Why xarray?

Metadata provides context and makes code more legible. This reduces the likelihood of errors from typos and makes analysis more intuitive (and fun!).

### Quick quiz

Analysis without xarray: what did this just average over?

 - A. Time
 - B. Latitude
 - C. Longitude
 - D. I can't tell

In [ ]:
temp = ds.tas.data  # just the NumPy array
temp.mean(axis=1).shape

The answer is latitude, since the dimensions are `(time, lat, lon)`. But you can't tell by looking at the line, and neither can anyone reviewing it.

### Analysis with xarray

Use dimension names instead of axis numbers:

In [ ]:
ds.tas.mean(dim='lat')

## Extracting data or "indexing"

Xarray supports

- label-based indexing using `.sel`
- position-based indexing using `.isel`

In [ ]:
ds.tas.sel(time=2100).plot()

Slicing and "nearest" indexing work too. Here's the temperature change for San Diego (32.7°N, 117.2°W, which is 242.8°E):

In [ ]:
san_diego = ds.tas.sel(lat=32.7, lon=242.8, method='nearest')
san_diego.sel(time=slice(1950, 2100)).plot()

Position-based indexing is similar to your usual NumPy `array[0, 2, 3]`, but with the power of named dimensions: `ds.tas.isel(time=0, lat=2, lon=3)`.

## Computation: weighting

Suppose we want the global mean temperature. Every grid cell is 2.5° of longitude wide, but the cells get much smaller towards the poles. The area of a cell is proportional to the cosine of its latitude:

In [ ]:
weights = np.cos(np.deg2rad(ds.lat))
weights

Note, NumPy functions like `np.cos` work on xarray objects and give you an xarray object back.

### Quick quiz

Will the area-weighted global mean temperature change in 2100 be higher or lower than the simple (unweighted) mean?

 - A. Higher
 - B. Lower
 - C. The same

In [ ]:
tas_2100 = ds.tas.sel(time=2100)
print('unweighted:', float(tas_2100.mean()))
print('weighted:  ', float(tas_2100.weighted(weights).mean()))

The Arctic warms much faster than the rest of the planet, and without weights every tiny polar grid cell counts as much as a big tropical one. So the unweighted mean overstates the warming by about a quarter of a degree.

Both numbers look perfectly plausible, so this is exactly the sort of mistake that's easy to miss.

### Broadcasting by name

`weights` has one dimension (`lat`) and `ds.tas` has three (`time`, `lat`, `lon`). Xarray broadcasts by dimension *name*, so there's no need to worry about the order of the axes or adding size-1 dimensions like we did with `keepdims` in NumPy:

In [ ]:
global_mean = ds.tas.weighted(weights).mean(['lat', 'lon'])

fig, ax = plt.subplots(figsize=(6, 3))
global_mean.plot(ax=ax, label='weighted')
ds.tas.mean(['lat', 'lon']).plot(ax=ax, label='unweighted')
ax.legend()

### Alignment

Like pandas, xarray lines data up by its coordinate labels before doing arithmetic. Here's a copy of the weights where the latitudes are off by a tiny amount (as can happen when two files store their grids slightly differently):

In [ ]:
weights_bad = weights.copy()
weights_bad['lat'] = weights.lat + 1e-5

### Quick quiz

What is the shape of `ds.tas.sel(time=2100) * weights_bad`?

 - A. `(96, 144)`
 - B. `(144,)`
 - C. `(0, 144)`
 - D. An error

In [ ]:
(ds.tas.sel(time=2100) * weights_bad).shape

None of the latitudes matched, so the result is empty, with no error. If you notice extra `NaN`s or missing points after an xarray computation, it usually means your coordinates weren't aligned *exactly*.

## High level computation

Xarray has very similar tools to pandas for common computations: `groupby`, `resample`, `rolling` and `coarsen`. For example, we can group by the country each grid cell is in:

In [ ]:
country = xr.open_dataarray('../data/country_mask.nc').rename('country')

warming_by_country = ds.tas.sel(time=2100).groupby(country).mean()
warming_by_country.to_series().nlargest(5)

Note, `to_series()` (and `to_dataframe()`) convert to pandas, and `xr.Dataset.from_dataframe()` goes the other way.

## Reading and writing files

Xarray supports many disk formats; netCDF and Zarr are the most common. For example:

```python
ds.to_netcdf('my-example-dataset.nc')
ds = xr.open_dataset('my-example-dataset.nc')
```

*Tip: `xr.open_mfdataset()` opens a collection of many files as one dataset, and with [Dask](https://www.dask.org/) it can work with data that is much bigger than memory.*

## Summary

Today, we covered the following topics:
 - Why we visualize data
 - matplotlib figures, axes and subplots (and why not to mix `plt.` and `ax.`)
 - Plotting with pandas
 - xarray DataArrays and Datasets: dimensions, coordinates and attributes
 - Indexing with `sel` and `isel`
 - Weighting, broadcasting by name and alignment

That's the end of Module A. On Monday (on Zoom), we'll start looking at how LLMs and coding agents actually work.